# Index Movies with Vector Embeddings

This notebook demonstrates how to index movie data into OpenSearch with vector embeddings using Amazon Bedrock.

## 1. Import Libraries

In [ ]:
import boto3
import json
import sys
sys.path.append('..')

from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
from utils.bedrock_embeddings import BedrockEmbeddings

## 2. Load Configuration

In [ ]:
# Load OpenSearch config
with open('../config.json', 'r') as f:
    config = json.load(f)

print("Configuration loaded:")
print(f"  Collection: {config['collection_name']}")
print(f"  Region: {config['region']}")
print(f"  Endpoint: {config['endpoint']}")

## 3. Initialize Bedrock Embeddings

In [ ]:
# Initialize embedding model
embedder = BedrockEmbeddings(region_name=config['region'])
model_info = embedder.get_model_info()

print("Embedding Model:")
print(f"  Model ID: {model_info['model_id']}")
print(f"  Provider: {model_info['provider']}")
print(f"  Dimension: {model_info['embedding_dimension']}")

## 4. Connect to OpenSearch

In [ ]:
# Extract host from endpoint
endpoint = config['endpoint']
host = endpoint.replace('https://', '').replace('http://', '')

# Get AWS credentials
credentials = boto3.Session().get_credentials()
auth = AWSV4SignerAuth(credentials, config['region'], 'aoss')

# Create OpenSearch client
client = OpenSearch(
    hosts=[{'host': host, 'port': 443}],
    http_auth=auth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=30
)

print("✓ Connected to OpenSearch")

## 5. Create Index with Vector Field

In [ ]:
index_name = 'movies'
embedding_dim = embedder.get_embedding_dimension()

index_body = {
    "settings": {
        "index": {
            "knn": True,
            "knn.algo_param.ef_search": 512
        }
    },
    "mappings": {
        "properties": {
            "id": {"type": "keyword"},
            "title": {"type": "text"},
            "year": {"type": "integer"},
            "genre": {"type": "keyword"},
            "director": {"type": "text"},
            "actors": {"type": "text"},
            "plot": {"type": "text"},
            "rating": {"type": "float"},
            "runtime": {"type": "integer"},
            "plot_embedding": {
                "type": "knn_vector",
                "dimension": embedding_dim,
                "method": {
                    "name": "hnsw",
                    "space_type": "cosinesimil",
                    "engine": "faiss",
                    "parameters": {
                        "ef_construction": 512,
                        "m": 16
                    }
                }
            },
            "combined_text": {"type": "text"}
        }
    }
}

# Delete if exists
if client.indices.exists(index=index_name):
    print(f"Deleting existing index '{index_name}'...")
    client.indices.delete(index=index_name)

# Create index
print(f"Creating index '{index_name}'...")
response = client.indices.create(index=index_name, body=index_body)
print("✓ Index created successfully")

## 6. Load Movie Data

In [ ]:
# Load movies from JSON
with open('../data/sample-movies-small.json', 'r') as f:
    movies = json.load(f)

print(f"Loaded {len(movies)} movies")
print(f"\nSample movie:")
print(json.dumps(movies[0], indent=2))

## 7. Generate Embeddings

In [ ]:
def prepare_movie_text(movie):
    """Combine movie fields for embedding"""
    parts = [
        f"Title: {movie['title']}",
        f"Plot: {movie['plot']}",
        f"Genre: {', '.join(movie['genre']) if isinstance(movie['genre'], list) else movie['genre']}",
        f"Director: {movie['director'] if isinstance(movie['director'], str) else ', '.join(movie['director'])}",
        f"Actors: {', '.join(movie['actors']) if isinstance(movie['actors'], list) else movie['actors']}"
    ]
    return " | ".join(parts)

# Prepare texts
print("Preparing movie texts...")
movie_texts = [prepare_movie_text(m) for m in movies]

# Generate embeddings
print("Generating embeddings with Bedrock...")
embeddings = embedder.embed_batch(
    movie_texts,
    input_type="search_document",
    batch_size=10
)

print(f"✓ Generated {len(embeddings)} embeddings")
print(f"  Embedding dimension: {len(embeddings[0])}")

## 8. Index Documents

In [ ]:
print(f"Indexing {len(movies)} movies...")
success_count = 0
error_count = 0

for i, (movie, embedding) in enumerate(zip(movies, embeddings)):
    try:
        doc = {
            **movie,
            'plot_embedding': embedding,
            'combined_text': movie_texts[i]
        }
        
        client.index(index=index_name, body=doc)
        success_count += 1
        
        if (i + 1) % 5 == 0:
            print(f"  Indexed {i + 1}/{len(movies)} movies...")
    
    except Exception as e:
        print(f"❌ Error indexing {movie.get('title', 'Unknown')}: {e}")
        error_count += 1

print(f"\n✅ Indexing complete!")
print(f"  Success: {success_count}")
print(f"  Errors: {error_count}")

## 9. Verify Index

In [ ]:
# Wait for documents to be searchable
import time
print("Waiting for documents to be searchable...")
time.sleep(5)

# Test search
response = client.search(
    index=index_name,
    body={"query": {"match_all": {}}, "size": 3}
)

print(f"\n📊 Index Verification:")
print(f"  Total documents: {response['hits']['total']['value']}")
print(f"\nSample documents:")
for hit in response['hits']['hits']:
    src = hit['_source']
    print(f"  - {src['title']} ({src['year']}) - ⭐ {src['rating']}")

## 10. Test Semantic Search

In [ ]:
# Generate query embedding
test_query = "movies about space exploration"
query_embedding = embedder.embed_text(test_query, input_type="search_query")

# Search
search_body = {
    "size": 3,
    "query": {
        "script_score": {
            "query": {"match_all": {}},
            "script": {
                "source": "knn_score",
                "lang": "knn",
                "params": {
                    "field": "plot_embedding",
                    "query_value": query_embedding,
                    "space_type": "cosinesimil"
                }
            }
        }
    },
    "_source": {"excludes": ["plot_embedding"]}
}

response = client.search(index=index_name, body=search_body)

print(f"\n🔍 Test Query: '{test_query}'")
print(f"\nTop Results:")
for i, hit in enumerate(response['hits']['hits'], 1):
    src = hit['_source']
    score = hit['_score']
    print(f"{i}. {src['title']} ({src['year']}) - ⭐ {src['rating']}")
    print(f"   Relevance: {score:.3f}")

print("\n" + "="*70)
print("✅ Indexing Complete!")
print("="*70)
print("\nNext: Run the Streamlit app or notebook 03_Test_Search.ipynb")